# Homework 5 — Analysing two weeks of plays

About 60 minutes. You analyse **all 14 days** of raw play events (1-14 September) with Spark.

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

### Exercise 1 — Load and clean
1. Read **all** files `DATA / "raw" / "plays_*.csv"` with the schema below.
2. Remove duplicate events (by `event_id`) and rows with `ms_played < 0`.
3. Add a column `play_date` (`F.to_date("event_ts")`).

Store the result in `plays`.

In [ ]:
PLAYS_SCHEMA = (
    "event_id STRING, event_ts TIMESTAMP, user_id STRING, track_id STRING, ms_played INT, "
    "device STRING, country STRING, skipped BOOLEAN, shuffle BOOLEAN"
)

# YOUR CODE HERE

print(plays.count(), "plays on", plays.select("play_date").distinct().count(), "days")

### Exercise 2 — Device share
For each device: number of plays and `share` = percent of all plays (rounded to 1 decimal).
Hint: compute `total = plays.count()` first and use it in `withColumn`.

In [ ]:
# YOUR CODE HERE

### Exercise 3 — When do people listen?
`F.hour(ts)` gives the hour, `F.dayofweek(ts)` gives 1 (Sunday) ... 7 (Saturday).
1. Add `hour` and `is_weekend` (`True` for Saturday and Sunday).
2. Count plays per `is_weekend` and `hour`.
3. Show the 3 busiest hours on weekdays and the 3 busiest on weekends. Are they different?

In [ ]:
# YOUR CODE HERE

### Exercise 4 — Does shuffle make people skip?
Compute the skip rate (percent, 1 decimal) for `shuffle = true` and `shuffle = false`.

In [ ]:
# YOUR CODE HERE

### Exercise 5 — Explicit content by plan
Load `tracks.csv` and `users.csv`. For each `plan`, what percentage of plays are of **explicit** tracks?

In [ ]:
# YOUR CODE HERE

### Exercise 6 — Save a result table
1. Build `daily_country`: per `play_date` and `country`: `plays` and `hours` (rounded to 2 decimals).
2. Write it as Parquet to `DATA.parent / "output" / "daily_country"`, **partitioned by country**, with mode `overwrite`.
3. Read back only Austria (`country=AT` sub-folder or filter after reading) and show it sorted by date.

In [ ]:
# YOUR CODE HERE

### Exercise 7 (Stretch) — Top 3 tracks per genre
Show, for every genre, the 3 most played tracks with their title and artist name. Use a window function with `F.row_number()`.

In [ ]:
# YOUR CODE HERE